# Sıfırdan Dil Modeli · Bölüm 2
## BPE: kelimeyi parçalara bölmek

Geçen bölümdeki tokenizer kelime kelime çalışıyordu: tanımadığı her kelime `<|unk|>` oluyor, Türkçenin ekleri sözlüğü şişiriyordu. Bu defterde GPT modellerinin kullandığı yöntemi, **BPE**'yi (*byte pair encoding*, bayt çifti kodlaması) sıfırdan yazıyoruz: metni baytlara ayır, en sık yan yana duran çifti birleştir, tekrarla.

Adımlar Sebastian Raschka'nın *Build a Large Language Model (From Scratch)* kitabının 2.5. bölümünü ve kitabın deposundaki "BPE from scratch" ekini izler. Kod, kitabın açık kaynak deposundaki kodun ([rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Apache-2.0) Türkçe metin için uyarlamasıdır.

Hücreleri sırayla çalıştırın: hücreye tıklayın, **Shift + Enter**'a basın.

## 1. GPT-2'nin tokenizer'ı
Önce hazır bir BPE tokenizer'ını deneyelim: GPT-2'ninkini. OpenAI'ın `tiktoken` kütüphanesi onu birkaç satırla kullanmamızı sağlıyor.

In [ ]:
!pip install -q tiktoken

import importlib.metadata
print("tiktoken sürümü:", importlib.metadata.version("tiktoken"))

In [ ]:
import tiktoken

gpt2 = tiktoken.get_encoding("gpt2")
print(gpt2.n_vocab)

Geçen bölümde tokenizer'ımızı çökerten cümleyi verelim. `<|endoftext|>` özel bir parça; metnin içinde görünce onu özel parça saysın diye `allowed_special` ile izin veriyoruz.

In [ ]:
text = ("Merhaba, nasılsın? <|endoftext|> "
        "Hasan'la Dadaruh dere kenarına inmişlerdi.")
ids = gpt2.encode(text, allowed_special={"<|endoftext|>"})
print(ids)
print(gpt2.decode(ids))

Her numara hangi parçaya karşılık geliyor? Numaraları tek tek çözelim:

In [ ]:
print([gpt2.decode([i]) for i in ids])

GPT-2 ağırlıkla İngilizce metinle eğitildi. Türkçe bir cümlede ne kadar parça kullanıyor?

In [ ]:
text = "Gümüş gibi parlayan bu güzel kaşağının dişlerine baktım."
ids = gpt2.encode(text)
print(len(text.split()), "kelime →", len(ids), "token")
print([gpt2.decode([i]) for i in ids])

## 2. Baytlar
Bir de tek bir kelime deneyelim:

In [ ]:
ids = gpt2.encode("İstanbul")
print(ids)
print([gpt2.decode([i]) for i in ids])

`�` işareti, tek başına harf olmayan bir bayt demek. Bilgisayar her harfi bir ya da birkaç **bayt** olarak saklar; bir bayt 0 ile 255 arasında bir sayıdır. Metni sayılara çevirmenin bu kuralına **UTF-8** denir.

In [ ]:
for ch in ["a", "ğ", "İ", "🙂"]:
    print(ch, list(ch.encode("utf-8")))

Öykümüzü yeniden indirip okuyalım (geçen bölümdeki iki hücre, tek hücrede):

In [ ]:
import os
import urllib.request

file_path = "kasagi.txt"
url = ("https://raw.githubusercontent.com/kocahmet1/sifirdan-dil-modeli/"
       "main/veri/kasagi.txt")
if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

Bütün öyküyü baytlara çevirelim. Her bayt 0–255 arasında bir sayı; yani 256 parçalık bir sözlük, her metni yazabilir.

In [ ]:
ids = list(raw_text.encode("utf-8"))
print("Karakter:", len(raw_text))
print("Bayt:", len(ids))
print(ids[:24])

## 3. En sık çifti saymak
BPE'nin fikri: yan yana en sık duran iki parçayı bul, onları tek bir yeni parçaya birleştir, tekrarla. Önce metni kelimelere ayırıyoruz ki birleştirmeler kelime sınırını aşmasın. GPT-2 de böyle yapar. Desen şunları ayırır: önünde en fazla bir boşluk olan harf dizileri, rakam dizileri, noktalama işaretleri ve boşluklar.

In [ ]:
import re

PAT = re.compile(r" ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+")
print(PAT.findall("Hasan'la Dadaruh dere kenarına inmişlerdi."))

Bütün öyküyü kelimelere ayırıp her kelimeyi bayt listesine çeviriyoruz:

In [ ]:
words = PAT.findall(raw_text)
chunks = [list(w.encode("utf-8")) for w in words]
print(len(chunks))
print(words[3:6])
print(chunks[3:6])

Şimdi her kelimenin içinde yan yana duran bütün çiftleri sayıyoruz. `zip(ids, ids[1:])` bir listeyi bir kaydırılmışıyla eşleştirir: (1., 2.), (2., 3.), … `Counter` her çiftin kaç kez geçtiğini sayar.

In [ ]:
from collections import Counter

def get_stats(chunks):
    counts = Counter()
    for ids in chunks:
        for pair in zip(ids, ids[1:]):
            counts[pair] += 1
    return counts

stats = get_stats(chunks)
print(stats.most_common(5))

Bu bayt çiftleri hangi harfler?

In [ ]:
for pair, count in stats.most_common(5):
    print(pair, repr(bytes(pair).decode("utf-8")), count)

## 4. Birleştirmek
`merge`, bir listede verilen çiftin geçtiği her yeri tek bir yeni numarayla değiştirir. 0–255 baytlar, 256 metin sonu (`<|endoftext|>`) için ayrıldı; yeni parçalar 257'den başlıyor.

In [ ]:
def merge(ids, pair, new_id):
    out = []
    i = 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

word = list("kaşağı".encode("utf-8"))
print(word)
print(merge(word, (196, 177), 257))

İlk birleştirmeyi bütün öyküye uygulayalım; toplam parça sayısı ne kadar azalıyor?

In [ ]:
merged = [merge(ids, (196, 177), 257) for ids in chunks]
print("Önce: ", sum(len(ids) for ids in chunks))
print("Sonra:", sum(len(ids) for ids in merged))

## 5. Eğitim: say, birleştir, tekrarla
Aynı iki adımı 500 kez tekrarlıyoruz. `merges` hangi çiftin hangi yeni numara olduğunu, `vocab` her numaranın hangi baytlardan oluştuğunu tutar. Tokenizer'ın "eğitimi" bu kadar.

In [ ]:
num_merges = 500
merges = {}
vocab = {i: bytes([i]) for i in range(256)}
vocab[256] = b"<|endoftext|>"

for i in range(num_merges):
    stats = get_stats(chunks)
    pair = max(stats, key=stats.get)
    new_id = 257 + i
    chunks = [merge(ids, pair, new_id) for ids in chunks]
    merges[pair] = new_id
    vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]

print("Sözlük:", len(vocab))
print("Öykü artık", sum(len(ids) for ids in chunks), "token")

Öğrendiği ilk 30 parça, öğrenme sırasıyla:

In [ ]:
print([vocab[i].decode("utf-8", errors="replace") for i in range(257, 287)])

Ve son öğrendikleri:

In [ ]:
print([vocab[i].decode("utf-8", errors="replace") for i in range(737, 757)])

## 6. Tokenizer sınıfı
Hepsini bir sınıfta topluyoruz. `encode` metni aynı desenle kelimelere, her kelimeyi baytlara ayırır; sonra kelimedeki çiftlerden öğrenilmiş olanlara bakar ve en önce öğrenileni (numarası en küçük olanı) birleştirir, öğrenilmiş çift kalmayana kadar. `decode` her numaranın baytlarını art arda ekleyip yeniden metne çevirir.

In [ ]:
class BPETokenizer:
    def __init__(self, merges):
        self.merges = merges
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.vocab[256] = b"<|endoftext|>"
        for (a, b), new_id in merges.items():
            self.vocab[new_id] = self.vocab[a] + self.vocab[b]

    def encode(self, text):
        out = []
        for word in PAT.findall(text):
            ids = list(word.encode("utf-8"))
            while len(ids) >= 2:
                pairs = [p for p in zip(ids, ids[1:]) if p in self.merges]
                if not pairs:
                    break
                pair = min(pairs, key=self.merges.get)
                ids = merge(ids, pair, self.merges[pair])
            out.extend(ids)
        return out

    def decode(self, ids):
        data = b"".join(self.vocab[i] for i in ids)
        return data.decode("utf-8", errors="replace")

In [ ]:
tokenizer = BPETokenizer(merges)

text = "Merhaba, nasılsın? Hasan'la Dadaruh dere kenarına inmişlerdi."
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

In [ ]:
print([tokenizer.decode([i]) for i in ids])
print(tokenizer.decode(ids) == text)

## 7. Türkçe
Geçen bölümde "ev" ailesinin yedi kelimesi de `<|unk|>` olmuştu. Bakalım BPE ne yapıyor:

In [ ]:
words = ["ev", "evler", "evlerimiz", "evlerimizde",
         "evlerimizdeki", "evlerimizdekiler", "evlerimizdekilerden"]

for word in words:
    pieces = [tokenizer.decode([i]) for i in tokenizer.encode(word)]
    print(f"{word:20} → {pieces}")

Bizim tokenizer'ımız yalnızca bir öyküden öğrendi. Aynı kodu çok daha büyük bir metinde çalıştırınca ne olur? Serinin modeli **Çırak**'ın tokenizer'ı aynı kuralla, Türkçe Vikipedi'den 20 MB metinde, 3.839 birleştirmeyle eğitildi. Birleştirmelerini indirip kendi sınıfımıza yüklüyoruz:

In [ ]:
import json

file_path = "cirak_tokenizer.json"
url = ("https://raw.githubusercontent.com/kocahmet1/sifirdan-dil-modeli/"
       "main/veri/cirak_tokenizer.json")
if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)

with open(file_path, encoding="utf-8") as f:
    data = json.load(f)
cirak_merges = {tuple(p): 257 + i for i, p in enumerate(data["merges"])}
cirak = BPETokenizer(cirak_merges)
print(len(cirak.vocab))

In [ ]:
for word in ["İstanbul", "kaşağının", "evlerimizdekilerden"]:
    print(word, "→", [cirak.decode([i]) for i in cirak.encode(word)])

Son olarak bütün öykü: kaç token tutuyor? (Bizim tokenizer'ımız bu öyküde eğitildiği için ona göre biraz avantajlı.)

In [ ]:
print("Bayt: ", len(raw_text.encode("utf-8")))
print("GPT-2:", len(gpt2.encode(raw_text)))
print("Bizim:", len(tokenizer.encode(raw_text)))
print("Çırak:", len(cirak.encode(raw_text)))

## Özet
BPE metni baytlardan başlatır; bu yüzden hiçbir zaman "bilinmeyen kelime" olmaz. Sonra en sık yan yana duran çiftleri birleştire birleştire, kimse ona öğretmeden, kelime parçalarını kendisi bulur: Türkçe için kökler ve ekler.

**Kendin dene**
- `num_merges`'i 100 ve 1000 yapıp eğitimi yeniden çalıştırın: "evlerimizdekilerden" kaç parçaya bölünüyor?
- Kendi uydurduğunuz uzun bir Türkçe kelimeyi üç tokenizer'a da verin: GPT-2, bizimki ve Çırak'ınki.

**Sıradaki bölüm:** kısa bir ara bölüm, PyTorch'a hızlı giriş.

---
Kaynak: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Manning 2024, bölüm 2.5 ve depodaki "BPE from scratch" eki. Kod: [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (Apache-2.0), Türkçe uyarlama: Sıfırdan Dil Modeli. Metin: Ömer Seyfettin, "Kaşağı" (kamu malı). Çırak'ın tokenizer'ı Türkçe Vikipedi metniyle (CC BY-SA 4.0) eğitildi.